# GLM–BG and RF–BG: Bernoulli–Gamma baselines for probabilistic daily precipitation downscaling

## Notation

| Symbol | Meaning | In the code |
|---|---|---|
| $\mathbf{x}_t\in\mathbb{R}^m$ | large-scale predictors of day $t$ ($m=8$), standardized | `s$X_standardized` |
| $y_t$ | precipitation at the station on day $t$ (mm), with the amounts $\le\tau$ set to 0 | `s$y_all` |
| $\tau$ | wet-day threshold, 1 mm | `CFG$tau` |
| $\delta_t=\mathbf{1}\{y_t>\tau\}$ | wet-day indicator | `s$wet_all` |
| $y_t-\tau$ | wet-day excess (0 on dry days) | `s$excess_all` |
| $p_t$, $\alpha_t$, $\beta_t$ | Bernoulli–Gamma parameters: wet-day probability, shape and rate of the Gamma distribution of the excess | columns `p`, `alpha`, `beta` |
| $\mu_t=\alpha_t/\beta_t$ | mean wet-day excess | |
| $N$ | ensemble members on each test day, 1000 | `CFG$ensemble_members` |

## 1. Setup

### 1.1 Environment and reproducibility

In [ ]:
Sys.setenv(OMP_NUM_THREADS = "1", OPENBLAS_NUM_THREADS = "1", MKL_NUM_THREADS = "1", VECLIB_MAXIMUM_THREADS = "1", NUMEXPR_NUM_THREADS = "1")

suppressPackageStartupMessages({
  library(parallel)
  library(reticulate)
  library(VGAM)
  library(RandomForestDist)
})
np <- import("numpy", convert = FALSE)

### 1.2 Configuration

| Setting | Element of `CFG` | Value |
|---|---|---|
| Wet-day threshold $\tau$ | `tau` | 1 mm |
| Predictors | `predictors` | the eight daily ERA5 means of Table 2 |
| Models and training scales | `families`, `time_scales`, `seasons` | GLM and RF; Annual, SeasonalConcat (DJF, MAM, JJA, SON), MonthConcat |
| Fitting and test periods | `train_start`, …, `test_end` | 1980–2017 and 2018–2025 |
| GLM–BG | `glm_stepsize`, `glm_maxit` | Fisher scoring with a step size of 0.5, at most 500 iterations |
| RF–BG | `rf_ntree`, `rf_mtry`, `rf_minbucket`, `rf_minsplit`, `rf_maxdepth` | 500 trees, 6 candidate predictors per split, leaves of at least 20 days, nodes of at least 60 days to be split, depth at most 15 |
| Ensemble | `ensemble_members` | 1000 members on each test day |
| Seed of the run | `seed` | 42 |

**The reasons for these values.**

- **Hyperparameters of RF–BG.** They were selected on the validation period with `tune_rf_parallel.R`: forests of 200
  trees were trained on 1980–2012 at the Annual scale and scored by the fair CRPS of 1000 members on 2013–2017. A
  first stage searched 2, 3, 4, 6 and 8 candidate predictors and minimum leaf sizes of 5, 10, 20 and 50, with a
  maximum depth of 10; a second stage tested maximum depths of 8, 15 and 30 for the best pair. The minimum split size
  is always three times the minimum leaf size. The configuration with the lowest validation CRPS, averaged over the
  stations, is used at all stations and training scales.
- **No model selection for GLM–BG.** The GLM has no hyperparameter, so both baselines are fitted on 1980–2017, five
  years more than the neural models.
- **One seed per random draw.** `seed_of` hashes the seed of the run and a list of names (station, model, group, tree)
  into an integer. Every tree and every ensemble thus has its own seed, which does not depend on the order in which
  the trees or the stations are processed.

In [ ]:
CFG <- list(
  tau = 1.0,
  seed = 42L,
  ensemble_members = 1000L,
  predictors = c("d2m_mean", "t2m_mean", "msl_mean", "tcwv_mean", "u850_mean", "v850_mean", "z850_mean", "q850_mean"),
  families = c("GLM", "RF"),
  time_scales = c("Annual", "SeasonalConcat", "MonthConcat"),
  seasons = list(DJF = c(12L, 1L, 2L), MAM = 3L:5L, JJA = 6L:8L, SON = 9L:11L),
  train_start = as.Date("1980-01-01"),
  train_end = as.Date("2017-12-31"),
  test_start = as.Date("2018-01-01"),
  test_end = as.Date("2025-12-31"),
  glm_maxit = 500L,
  glm_stepsize = 0.5,
  rf_ntree = 500L,
  rf_mtry = 6L,
  rf_minbucket = 20L,
  rf_minsplit = 60L,
  rf_maxdepth = 15L,
  rf_predict_chunk = 128L,
  # Station and R workers of the notebook
  station = "tours",
  workers = parallel::detectCores()
)
# Folders: the root of the package holds data/ and results/
parents <- function(folder) if (dirname(folder) == folder) folder else c(folder, parents(dirname(folder)))
root <- Find(function(folder) dir.exists(file.path(folder, "data", "Data1Cell_06UTC")), parents(getwd()))
CFG$era5_path <- file.path(root, "data", "Data1Cell_06UTC", paste0("era5_daily_cell_", CFG$station, ".npz"))
CFG$precipitation_path <- file.path(root, "data", "Data1Cell_06UTC", paste0("ecad_rr_", CFG$station, "_1980_2025.npy"))
CFG$output_dir <- file.path(root, "results", "notebook_glm_rf_tours")
month_groups <- setNames(lapply(1:12, as.integer), sprintf("%02d", 1:12))

seed_of <- function(...) {
  h <- 0
  for (code in utf8ToInt(paste(CFG$seed, ..., sep = "|"))) h <- (h * 31 + code) %% 2147483629
  as.integer(h)
}

describe <- function(value) if (is.list(value)) paste(names(value), collapse = ", ") else paste(format(value), collapse = ", ")
data.frame(value = vapply(CFG[!names(CFG) %in% c("era5_path", "precipitation_path", "output_dir")], describe, ""))

## 2. Problem and data

We observe a paired sequence $\{(\mathbf{x}_t,y_t)\}_{t=1}^{T}$ at one station:

- $\mathbf{x}_t\in\mathbb{R}^m$ holds the $m=8$ daily predictors of the ERA5 reanalysis (Hersbach et al., 2020) for
  the grid cell that contains the station;
- $y_t$ is the precipitation measured at the station on day $t$, from ECA&D (Klein Tank et al., 2002).

Amounts at or below the wet-day threshold $\tau=1$ mm are set to zero, so that $y_t\in\{0\}\cup(\tau,\infty)$, and
$\delta_t=\mathbf{1}\{y_t>\tau\}$ marks the wet days.

The two baselines are **same-day models**: the Bernoulli–Gamma parameters of day $t$ are functions of the predictors
of that day only,

$$
(p_t,\alpha_t,\beta_t)=\boldsymbol\eta(\mathbf{x}_t),\qquad
p(y_t\mid\mathbf{x}_t)=\mathrm{BG}(y_t;\,p_t,\alpha_t,\beta_t),
$$

so that the predictive distribution of each day is one Bernoulli–Gamma distribution, and the days are independent
given the predictors. They use no observed precipitation of the target period, and no temporal dependence other than
that of the predictors.

**Splits.** GLM–BG has no model selection, and the hyperparameters of RF–BG were selected beforehand (Section 1.2), so
both models are fitted on 1980–2017 and evaluated on 2018–2025.

| Period | Days | Use |
|---|---|---|
| 1980–2017 | 13 880 | fitting |
| 2018–2025 | 2922 | test: evaluation only |

**Scaling.** Each predictor is standardized with the mean and the standard deviation of 1980–2017, the fitting period
(`sd`, with the denominator $n-1$). The same transformation is applied to the test days.

### 2.1 The station and the data

**Tours** lies in the Loire valley, in west-central France, with a climate transitional between the oceanic regime of
the west and the continental regime of the north-east. The eight predictors of Table 2
are ERA5 means over the 24 hours of the ECA&D observation day, from 06 UTC to 06 UTC. The cell below reads the two
files and builds the list `s` of the station, as `run_glm_rf_parallel.R` does for every station.

In [ ]:
era5 <- np$load(CFG$era5_path, allow_pickle = FALSE)
X_all <- py_to_r(era5[["data"]])
dates <- as.Date(as.character(py_to_r(np$datetime_as_string(era5[["dates"]], unit = "D"))))
variable_names <- as.character(py_to_r(era5[["variables"]]))
invisible(era5$close())

X_raw <- X_all[, match(CFG$predictors, variable_names), drop = FALSE]
colnames(X_raw) <- CFG$predictors
storage.mode(X_raw) <- "double"
y_raw <- as.numeric(py_to_r(np$load(CFG$precipitation_path, allow_pickle = FALSE)))
stopifnot(length(y_raw) == nrow(X_raw), !anyNA(y_raw), !anyNA(X_raw))
y_all <- ifelse(y_raw > CFG$tau, y_raw, 0.0)
wet_all <- as.integer(y_all > CFG$tau)
excess_all <- ifelse(wet_all == 1L, y_all - CFG$tau, 0.0)
months <- as.integer(format(dates, "%m"))
train_rows <- which(dates >= CFG$train_start & dates <= CFG$train_end)
test_rows <- which(dates >= CFG$test_start & dates <= CFG$test_end)
x_mean <- colMeans(X_raw[train_rows, , drop = FALSE])
x_sd <- apply(X_raw[train_rows, , drop = FALSE], 2, sd)
x_sd[!is.finite(x_sd) | x_sd == 0] <- 1
X_standardized <- sweep(sweep(X_raw, 2, x_mean, "-"), 2, x_sd, "/")
colnames(X_standardized) <- CFG$predictors

s <- list(name = CFG$station, dates = dates, months = months, X_standardized = X_standardized, y_all = y_all, wet_all = wet_all,
          excess_all = excess_all, train_rows = train_rows, test_rows = test_rows, out_dir = CFG$output_dir)

# The station and the two periods
IRdisplay::display(data.frame(`ECA&D identifier (STAID)` = 2190L, `latitude (°N)` = 47.44, `longitude (°E)` = 0.73, `elevation (m)` = 108L,
                              `mean annual precipitation (mm)` = round(mean(tapply(y_raw, format(dates, "%Y"), sum))),
                              `wet days (%)` = round(100 * mean(wet_all), 1), row.names = "Tours", check.names = FALSE))
data.frame(split = c("fitting", "test"), first = dates[c(train_rows[1], test_rows[1])], last = dates[c(tail(train_rows, 1), tail(test_rows, 1))],
           days = c(length(train_rows), length(test_rows)),
           `wet days (%)` = round(100 * c(mean(wet_all[train_rows]), mean(wet_all[test_rows])), 1),
           `mean wet-day amount (mm)` = round(c(mean(y_all[train_rows][wet_all[train_rows] == 1L]), mean(y_all[test_rows][wet_all[test_rows] == 1L])), 2),
           check.names = FALSE)

### 2.2 Training scales

Each model is fitted at three training scales:

- **Annual** (`Annual`): one model, fitted on all the days of the year;
- **Season** (`SeasonalConcat`): four models, one per season (DJF, MAM, JJA, SON), each fitted on the days of its
  season;
- **Month** (`MonthConcat`): twelve models, one per calendar month.

Every test day is predicted by the model of its season or month. A finer scale lets the relation between the
predictors and the precipitation change through the year, but leaves fewer days, and fewer wet days, to fit each
model. `build_groups` returns the rows of every group; the table gives the numbers of fitting and test days.

In [ ]:
build_groups <- function(s, rows, time_scale) {
  month_sets <- if (time_scale == "Annual") list(ALL = 1L:12L) else if (time_scale == "SeasonalConcat") CFG$seasons else month_groups
  lapply(month_sets, function(group_months) rows[s$months[rows] %in% group_months])
}

chain_rows <- list()
for (time_scale in CFG$time_scales) for (split in c("train", "test")) {
  groups <- build_groups(s, if (split == "train") train_rows else test_rows, time_scale)
  for (group in names(groups)) chain_rows[[length(chain_rows) + 1L]] <- data.frame(time_scale = time_scale, split = split, group = group, days = length(groups[[group]]))
}
groups_table <- reshape(do.call(rbind, chain_rows), idvar = c("time_scale", "group"), timevar = "split", direction = "wide")
rownames(groups_table) <- NULL
groups_table

## 3. The Bernoulli–Gamma distribution of the baselines

Both baselines predict, for each day, the three parameters of the Bernoulli–Gamma distribution of Eq. (1): the
wet-day probability $p_t$, and the shape $\alpha_t$ and the rate $\beta_t$ of the Gamma distribution of the excess
$y_t-\tau$ on a wet day, whose mean is $\mu_t=\alpha_t/\beta_t$:

$$
p(y_t\mid p_t,\alpha_t,\beta_t)=(1-p_t)^{1-\delta_t}\left[p_t\,\frac{\beta_t^{\alpha_t}}{\Gamma(\alpha_t)}\,
(y_t-\tau)^{\alpha_t-1}\,e^{-\beta_t(y_t-\tau)}\right]^{\delta_t}.
$$

- **`sanitize_parameters`** keeps $p_t$ in $[0,1]$ and $\alpha_t$, $\beta_t$ above $10^{-8}$, so that a degenerate
  prediction cannot break the simulation or the scores.
- **`simulate_bg`** draws the ensemble. Member $n$ of day $t$ is wet when a uniform draw falls below $p_t$, and its
  amount is then $\tau$ plus a Gamma draw with shape $\alpha_t$ and rate $\beta_t$. The members of a day are
  independent draws from the same distribution.
- **Moments.** The predictive mean of day $t$ is $p_t(\tau+\mu_t)$.

In [ ]:
sanitize_parameters <- function(parameters) {
  parameters <- as.matrix(parameters[, c("p", "alpha", "beta"), drop = FALSE])
  storage.mode(parameters) <- "double"
  parameters[, "p"] <- pmin(1, pmax(0, parameters[, "p"]))
  parameters[, "alpha"] <- pmax(1e-8, parameters[, "alpha"])
  parameters[, "beta"] <- pmax(1e-8, parameters[, "beta"])
  parameters
}

simulate_bg <- function(parameters, members = CFG$ensemble_members) {
  parameters <- sanitize_parameters(parameters)
  n <- nrow(parameters)
  wet <- matrix(runif(n * members), nrow = n) < parameters[, "p"]
  excess <- matrix(rgamma(n * members, shape = rep(parameters[, "alpha"], members), rate = rep(parameters[, "beta"], members)), nrow = n)
  ifelse(wet, CFG$tau + excess, 0.0)
}

## 4. GLM–BG: generalized linear model

The generalized linear model links each Bernoulli–Gamma parameter to the predictors of the same day:

$$
\operatorname{logit}p_t=a_0+\mathbf{a}^\top\mathbf{x}_t,\qquad
\log\mu_t=b_0+\mathbf{b}^\top\mathbf{x}_t,\qquad
\log\alpha_t=c_0+\mathbf{c}^\top\mathbf{x}_t .
$$

- **Occurrence.** A logistic regression (`binomialff`, logit link) is fitted to $\delta_t$ on all the fitting days of
  the group.
- **Intensity.** A Gamma regression (`gamma2`, log links for the mean and the shape) is fitted to the excess
  $y_t-\tau$ on the wet days only. With `zero = NULL` and `parallel = FALSE`, the mean and the shape both depend on all
  the predictors, each with its own coefficients, so that all three parameters vary in time through $\mathbf{x}_t$.
- **Estimation.** Both regressions are fitted by maximum likelihood with the Fisher scoring of VGAM (Yee, 2015), here
  with a step size of 0.5 and at most 500 iterations for the Gamma regression.

`predict_glm_parameters` returns $p_t$ from the linear predictor of the occurrence model, $\mu_t$ and $\alpha_t$ from
the exponential of the two linear predictors of the intensity model, and $\beta_t=\alpha_t/\mu_t$.

In [ ]:
fit_glm_group <- function(s, rows) {
  occurrence_formula <- reformulate(CFG$predictors, response = "wet")
  intensity_formula <- reformulate(CFG$predictors, response = "excess")
  frame <- data.frame(wet = s$wet_all[rows], excess = s$excess_all[rows], s$X_standardized[rows, , drop = FALSE])
  wet_frame <- frame[frame$wet == 1L, , drop = FALSE]
  occurrence <- vglm(occurrence_formula, binomialff(link = "logitlink"), data = frame)
  intensity <- vglm(intensity_formula, gamma2(lmu = "loglink", lshape = "loglink", zero = NULL, parallel = FALSE), data = wet_frame, control = vglm.control(stepsize = CFG$glm_stepsize, maxit = CFG$glm_maxit))
  list(occurrence = occurrence, intensity = intensity, train_days = nrow(frame), train_wet_days = nrow(wet_frame))
}

predict_glm_parameters <- function(s, model, rows) {
  newdata <- data.frame(s$X_standardized[rows, , drop = FALSE])
  p <- plogis(drop(predict(model$occurrence, newdata = newdata, type = "link")))
  eta <- as.matrix(predict(model$intensity, newdata = newdata, type = "link"))
  mu <- exp(eta[, 1])
  alpha <- exp(eta[, 2])
  sanitize_parameters(cbind(p = p, alpha = alpha, beta = alpha / mu))
}

The coefficients of the Annual model show how each standardized predictor acts on the three parameters.

In [ ]:
glm_annual <- fit_glm_group(s, s$train_rows)
intensity <- coef(glm_annual$intensity)
round(data.frame(`logit p` = coef(glm_annual$occurrence), `log mu` = intensity[grep(":1$", names(intensity))],
                 `log alpha` = intensity[grep(":2$", names(intensity))], row.names = c("(Intercept)", CFG$predictors), check.names = FALSE), 3)

**Reading the table.** The predictors are standardized, so the coefficients measure the effect of one standard
deviation of each predictor.

## 5. RF–BG: a-posteriori random forest

The a-posteriori random forest was introduced for the Gamma distribution of wet-day amounts by Legasa et al. (2022)
and extended to the Bernoulli–Gamma distribution by Legasa et al. (2023). It is a nonlinear, same-day model,
implemented in the package RandomForestDist, which is built on a fork of rpart.

- **Trees.** Each of the 500 trees is grown on a bootstrap sample of the fitting days: as many days as in the group,
  drawn with replacement. At each node, `mtry = 6` of the 8 predictors are candidates, and the split maximizes the
  Bernoulli–Gamma log-likelihood of the two child nodes, with the parameters of each node estimated by moment matching
  (`method = "bernoulliGammaLLMME"`). A node is split only if it holds at least `minsplit = 60` days, every leaf keeps
  at least `minbucket = 20` days, and the depth is at most 15.
- **A posteriori estimation.** The trees store the excesses of the days in their leaves (`keep.y = TRUE`). For a test
  day, the excesses of the leaves that it reaches in the 500 trees are pooled, and $(p_t,\alpha_t,\beta_t)$ are
  estimated from this pooled sample (`estimateBernoulliGammabc3` of RandomForestDist, with the BC3 estimator of the
  Gamma parameters). The forest thus predicts a distribution, and not only a mean.
- **Seeds.** Every tree has its own seed, from the seed of the run, the station, the model, the group and the index of
  the tree (`seed_of`), so that a forest does not depend on how its trees are shared among the R workers. The rpart
  fork draws the candidate predictors with the C function `rand()`, which `set.seed` does not reach, so `srand()` is
  reseeded before every tree as well (`c_srand`).
- **Platform.** `rand()` is implemented differently on Linux and macOS. On another platform than that of the paper
  runs (Linux), the candidate predictors, and hence the trees, differ, and the scores of RF–BG differ slightly from
  those of the paper. GLM–BG uses no random numbers.

`run_glm_rf_parallel.R` compiles `c_srand` with `R CMD SHLIB`. The notebook calls the system C compiler instead,
because the compilers of some conda environments produce a library that recent macOS versions do not load.

In [ ]:
# srand() for the C rand() used by the rpart fork, compiled once
SRAND_LIB <- local({
  build_dir <- tempfile("c_srand_")
  dir.create(build_dir)
  writeLines(c("#include <stdlib.h>", "void c_srand(int *seed) { srand((unsigned int) *seed); }"), file.path(build_dir, "c_srand.c"))
  library_file <- file.path(build_dir, paste0("c_srand", .Platform$dynlib.ext))
  status <- system2("cc", c("-shared", "-fPIC", "-o", shQuote(library_file), shQuote(file.path(build_dir, "c_srand.c"))))
  if (status != 0) stop("Could not compile c_srand.c with cc")
  library_file
})
dyn.load(SRAND_LIB)

# One tree per randomForestTrain call, each with its own seed, so a forest does not depend on how it is cut into blocks.
fit_rf_trees <- function(s, rows, trees, seed_key) {
  x <- unname(s$X_standardized[rows, , drop = FALSE])
  y <- s$excess_all[rows]
  multiresponse <- NA
  forest <- lapply(trees, function(tree_index) {
    tree_seed <- seed_of(seed_key, "tree", tree_index)
    set.seed(tree_seed)
    .C("c_srand", tree_seed)  # the rpart fork draws the mtry variables with C rand(), which set.seed does not reach
    one <- randomForestTrain(x = x, y = y, ntree = 1L, mtry = CFG$rf_mtry, minbucket = CFG$rf_minbucket, minsplit = CFG$rf_minsplit, maxdepth = CFG$rf_maxdepth, method = "bernoulliGammaLLMME", resample = TRUE, replace = TRUE, parallel.plan = NA, keep.y = TRUE, progress.bar = FALSE)
    multiresponse <<- attr(one, "multiresponse")
    one[[1]]
  })
  list(trees = forest, multiresponse = multiresponse)
}

predict_rf_parameters <- function(s, model, rows) {
  estimate_bg_leaf <- getFromNamespace("estimateBernoulliGammabc3", "RandomForestDist")
  predictors <- unname(s$X_standardized[rows, , drop = FALSE])
  chunks <- split(seq_len(nrow(predictors)), ceiling(seq_len(nrow(predictors)) / CFG$rf_predict_chunk))
  parameters <- do.call(rbind, lapply(chunks, function(index) {
    leaves <- randomForestPredict(model$forest, newdata = predictors[index, , drop = FALSE], method = "leaves", parallel.plan = NA)
    t(vapply(leaves, function(values) estimate_bg_leaf(as.numeric(values), non_informative_beta = 1), numeric(3)))
  }))
  colnames(parameters) <- c("p", "alpha", "beta")
  sanitize_parameters(parameters)
}

**R workers.** As in the script, a cluster of R processes (`makePSOCKcluster`) fits the trees: each worker loads
RandomForestDist and `c_srand`, and fits a block of trees with `fit_rf_trees`. `fit_rf_forest` joins the blocks into
one forest, as `task_rf_predict` does in the script.

In [ ]:
cl <- makePSOCKcluster(CFG$workers)
rf_block <- function(trees, rows, seed_key) fit_rf_trees(s, rows, trees, seed_key)
clusterExport(cl, c("SRAND_LIB", "CFG", "s", "seed_of", "fit_rf_trees", "rf_block"))
invisible(clusterEvalQ(cl, {
  suppressPackageStartupMessages(library(RandomForestDist))
  dyn.load(SRAND_LIB)
  NULL
}))

fit_rf_forest <- function(rows, model_name, group) {
  # the trees of one forest, in one block per worker; every tree has its own seed (fit_rf_trees)
  blocks <- unname(split(seq_len(CFG$rf_ntree), cut(seq_len(CFG$rf_ntree), CFG$workers, labels = FALSE)))
  pieces <- parLapply(cl, blocks, rf_block, rows = rows, seed_key = paste(s$name, model_name, group, sep = "|"))
  structure(do.call(c, lapply(pieces, `[[`, "trees")), multiresponse = pieces[[1]]$multiresponse, class = "RandomForestDist")
}
cat(length(cl), "R workers ready\n")

## 6. Fitting the six systems at Tours

The loop fits the six systems of `run_glm_rf_parallel.R`: GLM–BG and RF–BG at the three training scales. For each
group, the model is fitted on the fitting days of the group and predicts the Bernoulli–Gamma parameters of its test
days. The parameters of the groups are then assembled into one series of test days per system, as `task_inputs` does
in the script. The fit log gives the numbers of fitting days and wet days, and the time of each group.

In [ ]:
parameters_by_model <- list()
fit_log <- list()
for (family in CFG$families) for (time_scale in CFG$time_scales) {
  model_name <- paste(family, time_scale, sep = "_")
  train_groups <- build_groups(s, s$train_rows, time_scale)
  test_groups <- build_groups(s, s$test_rows, time_scale)
  parameters <- matrix(NA_real_, nrow = length(s$test_rows), ncol = 3L, dimnames = list(NULL, c("p", "alpha", "beta")))
  started_model <- proc.time()[["elapsed"]]
  for (group in names(train_groups)) {
    started <- proc.time()[["elapsed"]]
    train <- train_groups[[group]]
    test <- test_groups[[group]]
    if (family == "GLM") {
      model <- fit_glm_group(s, train)
      group_parameters <- predict_glm_parameters(s, model, test)
    } else {
      model <- list(forest = fit_rf_forest(train, model_name, group))
      group_parameters <- predict_rf_parameters(s, model, test)
    }
    parameters[match(test, s$test_rows), ] <- group_parameters
    fit_log[[length(fit_log) + 1L]] <- data.frame(model = model_name, group = group, train_days = length(train), train_wet_days = sum(s$wet_all[train]),
                                                  test_days = length(test), seconds = round(proc.time()[["elapsed"]] - started, 1))
    rm(model); invisible(gc())
  }
  stopifnot(!anyNA(parameters))
  parameters_by_model[[model_name]] <- sanitize_parameters(parameters)
  cat(sprintf("%-18s %2d groups fitted in %5.0f s\n", model_name, length(train_groups), proc.time()[["elapsed"]] - started_model))
}
stopCluster(cl)
fit_log <- do.call(rbind, fit_log)

# Outputs: the parameters of the test days and the fit log
dir.create(CFG$output_dir, recursive = TRUE, showWarnings = FALSE)
parameter_table <- data.frame(date = s$dates[s$test_rows], do.call(cbind, lapply(names(parameters_by_model), function(model_name)
  setNames(as.data.frame(parameters_by_model[[model_name]]), paste(model_name, c("p", "alpha", "beta"), sep = "_")))))
write.csv(parameter_table, file.path(CFG$output_dir, "test_parameters.csv"), row.names = FALSE)
write.csv(fit_log, file.path(CFG$output_dir, "fit_log.csv"), row.names = FALSE)
time_by_model <- aggregate(cbind(groups = 1, seconds) ~ model, data = fit_log, FUN = sum)
time_by_model <- time_by_model[match(names(parameters_by_model), time_by_model$model), ]
rownames(time_by_model) <- NULL
time_by_model

## 7. Ensembles and evaluation on the test period 2018–2025

### 7.1 Ensembles

As in the script, each system simulates 1000 members on each test day with `simulate_bg`, with the seed
`seed_of(station, model, "ensemble")`. The paper runs store the members as 32-bit floating-point numbers
(`test_ensemble.npz`) and score the stored values; the notebook rounds them in the same way.

In [ ]:
ensembles <- list()
for (model_name in names(parameters_by_model)) {
  set.seed(seed_of(s$name, model_name, "ensemble"))
  ensemble <- simulate_bg(parameters_by_model[[model_name]])
  ensembles[[model_name]] <- py_to_r(np$asarray(ensemble, dtype = "float32")$astype("float64"))
}
observed <- s$y_all[s$test_rows]
sapply(ensembles, function(ensemble) c(days = nrow(ensemble), members = ncol(ensemble), `wet members (%)` = round(100 * mean(ensemble > CFG$tau), 1)))

### 7.2 Scores

Let $\mathcal{T}$ be the 2922 test days, $y_t$ the observation and $y^{(1)}_t,\ldots,y^{(N)}_t$ the members of day $t$. The scores are proper scoring rules or measures of discrimination (Gneiting and
Raftery, 2007):

- **CRPS**, from the ensemble with the fair estimator (Ferro, 2014) and, for comparison, from the
  Bernoulli–Gamma distribution in closed form, averaged over $\mathcal{T}$, in mm day$^{-1}$;
- **NLL**: $-\frac{1}{|\mathcal{T}|}\sum_t\log p(y_t\mid p_t,\alpha_t,\beta_t)$;
- **Brier score** and **AUC** of the wet-day probability $p_t$;
- **COR** and **RMSE** of the ensemble mean;
- **CRPSS** of the fair CRPS against the monthly climatology, whose 1000 members are drawn
  from the days of the same calendar month in 1980–2017.

In [ ]:
threshold_precipitation <- function(values, threshold = 1.0) ifelse(values > threshold, values, 0.0)

crps_ensemble_fair <- function(observed, ensemble, threshold = 1.0) {
  # daily fair ensemble CRPS; ensemble of shape (days, members)
  observed <- threshold_precipitation(observed, threshold)
  ensemble <- threshold_precipitation(ensemble, threshold)
  n_members <- ncol(ensemble)
  first_term <- rowMeans(abs(ensemble - observed))
  ordered <- t(apply(ensemble, 1, sort))
  coefficients <- 2.0 * seq_len(n_members) - n_members - 1.0
  first_term - drop(ordered %*% coefficients) / (n_members * (n_members - 1.0))
}

crps_bernoulli_gamma_analytic <- function(observed, wet_probability, alpha, beta, threshold = 1.0) {
  # daily CRPS of 0 or threshold + Gamma(alpha, rate = beta), in closed form
  observed <- threshold_precipitation(observed, threshold)
  wet_probability <- pmin(1, pmax(0, wet_probability))
  shifted_observed <- pmax(observed - threshold, 0.0)
  gamma_mean <- alpha / beta
  gamma_cdf <- pgamma(beta * shifted_observed, shape = alpha)
  truncated_first_moment <- gamma_mean * pgamma(beta * shifted_observed, shape = alpha + 1.0)
  expected_absolute_error <- shifted_observed * (2.0 * gamma_cdf - 1.0) + gamma_mean - 2.0 * truncated_first_moment
  mean_absolute_gamma_difference <- 2.0 * exp(lgamma(alpha + 0.5) - lgamma(alpha)) / (beta * sqrt(pi))
  half_expected_pair_distance <- wet_probability * (1.0 - wet_probability) * (threshold + gamma_mean) + 0.5 * wet_probability^2 * mean_absolute_gamma_difference
  crps_dry <- wet_probability * (threshold + gamma_mean) - half_expected_pair_distance
  crps_wet <- (1.0 - wet_probability) * observed + wet_probability * expected_absolute_error - half_expected_pair_distance
  ifelse(observed == 0.0, crps_dry, crps_wet)
}

bernoulli_gamma_log_likelihood <- function(observed, wet_probability, alpha, beta, threshold = 1.0) {
  # daily log predictive density of 0 or threshold + Gamma(alpha, rate = beta)
  observed <- threshold_precipitation(observed, threshold)
  wet_probability <- pmin(1 - 1e-12, pmax(1e-12, wet_probability))
  excess <- pmax(observed - threshold, 1e-12)
  log_wet <- log(wet_probability) + alpha * log(beta) - lgamma(alpha) + (alpha - 1.0) * log(excess) - beta * excess
  ifelse(observed > 0.0, log_wet, log1p(-wet_probability))
}

brier_score <- function(observed, wet_probability, threshold = 1.0) mean((pmin(1, pmax(0, wet_probability)) - (observed > threshold))^2)

auc_score <- function(observed, wet_probability, threshold = 1.0) {
  # ROC AUC for wet occurrence, computed from ranks
  target <- observed > threshold
  ranks <- rank(wet_probability)
  n_wet <- sum(target)
  n_dry <- sum(!target)
  (sum(ranks[target]) - n_wet * (n_wet + 1) / 2) / (n_wet * n_dry)
}

# Monthly climatology of every test day (Eq. A.7), drawn as in stations_summary.py
reference <- s$dates >= as.Date("1980-01-01") & s$dates <= as.Date("2017-12-31")
rng <- np$random$default_rng(0L)
draws <- lapply(1:12, function(m) py_to_r(rng$choice(np$array(s$y_all[reference][s$months[reference] == m]), size = 1000L, replace = TRUE)))
climatology <- do.call(rbind, draws[s$months[s$test_rows]])
crps_climatology <- crps_ensemble_fair(observed, climatology, CFG$tau)

daily_crps <- lapply(ensembles, function(ensemble) crps_ensemble_fair(observed, ensemble, CFG$tau))
scores <- do.call(rbind, lapply(names(parameters_by_model), function(model_name) {
  parameters <- parameters_by_model[[model_name]]
  ensemble_mean <- rowMeans(threshold_precipitation(ensembles[[model_name]], CFG$tau))
  data.frame(model = model_name, CRPS = mean(daily_crps[[model_name]]),
             `CRPS, closed form` = mean(crps_bernoulli_gamma_analytic(observed, parameters[, "p"], parameters[, "alpha"], parameters[, "beta"], CFG$tau)),
             NLL = -mean(bernoulli_gamma_log_likelihood(observed, parameters[, "p"], parameters[, "alpha"], parameters[, "beta"], CFG$tau)),
             Brier = brier_score(observed, parameters[, "p"], CFG$tau), AUC = auc_score(observed, parameters[, "p"], CFG$tau),
             COR = cor(observed, ensemble_mean), RMSE = sqrt(mean((ensemble_mean - observed)^2)),
             CRPSS = 1 - mean(daily_crps[[model_name]]) / mean(crps_climatology), check.names = FALSE)
}))
write.csv(scores, file.path(CFG$output_dir, "scores.csv"), row.names = FALSE)
cat(sprintf("CRPS of the monthly climatology: %.3f mm/day\n", mean(crps_climatology)))
data.frame(scores[1], round(scores[-1], 4), check.names = FALSE)

**Table 1.** Scores of GLM–BG and RF–BG over the test period 2018–2025, at the three training scales.

### 7.3 Monthly skill

For month $j$, $\mathrm{CRPSS}_j$ compares the mean fair CRPS of a system and of the climatology over the test days of
that month.

In [ ]:
test_months <- s$months[s$test_rows]
monthly <- sapply(daily_crps, function(crps) vapply(1:12, function(m) 1 - mean(crps[test_months == m]) / mean(crps_climatology[test_months == m]), 0))
rownames(monthly) <- month.abb

options(repr.plot.width = 10.5, repr.plot.height = 3.8, repr.plot.res = 110)
par(mfrow = c(1, 2), mar = c(3, 4.2, 2.2, 0.8), mgp = c(2.4, 0.7, 0), cex = 0.85)
for (family in CFG$families) {
  plot(NA, xlim = c(1, 12), ylim = range(c(monthly, 0)), xaxt = "n", xlab = "", ylab = "CRPSS against the monthly climatology",
       main = if (family == "GLM") "(a) GLM–BG" else "(b) RF–BG", adj = 0, font.main = 1)
  axis(1, at = 1:12, labels = month.abb)
  abline(h = 0, col = "grey45", lty = 2)
  for (k in seq_along(CFG$time_scales)) {
    lines(1:12, monthly[, paste(family, CFG$time_scales[k], sep = "_")], type = "o", pch = c(16, 15, 17)[k],
          col = c("#1a1a1a", "#2a78d6", "#dd5b25")[k], lwd = 1.4, cex = 0.8)
  }
  legend("bottomright", legend = c("Annual", "Season", "Month"), col = c("#1a1a1a", "#2a78d6", "#dd5b25"), pch = c(16, 15, 17),
         lwd = 1.4, bty = "n", cex = 0.9, y.intersp = 1.3, inset = c(0.02, 0.04))
}
round(t(monthly), 2)

**Figure 1.** Monthly CRPS skill score of (a) GLM–BG and (b) RF–BG against the monthly climatology, at the
three training scales. The dashed line marks the skill of the climatology.

## 8. References

- Ferro, C. A. T. (2014). Fair scores for ensemble forecasts. *Quarterly Journal of the Royal Meteorological Society*, 140.
- Gneiting, T. and Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *Journal of the
  American Statistical Association*, 102, 359–378. https://doi.org/10.1198/016214506000001437
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146,
  1999–2049. https://doi.org/10.1002/qj.3803
- Klein Tank, A. M. G. et al. (2002). Daily dataset of 20th-century surface air temperature and precipitation series
  for the European Climate Assessment. *International Journal of Climatology*, 22, 1441–1453.
  https://doi.org/10.1002/joc.773
- Legasa, M. N., Manzanas, R., Calviño, A. and Gutiérrez, J. M. (2022). A posteriori random forests for stochastic
  downscaling of precipitation by predicting probability distributions. *Water Resources Research*, 58(4),
  e2021WR030272. https://doi.org/10.1029/2021WR030272
- Legasa, M. N., Thao, S., Vrac, M. and Manzanas, R. (2023). Assessing three perfect prognosis methods for statistical
  downscaling of climate change precipitation scenarios. *Geophysical Research Letters*, 50(9), e2022GL102525.
  https://doi.org/10.1029/2022GL102525
- Yee, T. W. (2015). *Vector Generalized Linear and Additive Models: With an Implementation in R*. Springer, New York.